In [1]:
#importing the necessary libraries
import pandas as pd
import numpy as np

***Data Profiling***

*Traffic_Sensor_Log.csv*

In [2]:
t = pd.read_csv("../data/raw/traffic_sensor_log.csv")

In [3]:
# --- Structure ---
print(t.shape)
print(t.dtypes)
t.head()

(178468, 17)
road_id                   str
road_name                 str
latitude              float64
longitude             float64
weather_station_id        str
date                      str
time                      str
traffic_volume        float64
vehicle_count           int64
vehicle_type_dist         str
avg_speed             float64
occupancy             float64
congestion_level          str
travel_time           float64
accident_count          int64
signal_timing           int64
road_capacity           int64
dtype: object


,road_id,road_name,latitude,longitude,weather_station_id,date,time,traffic_volume,vehicle_count,vehicle_type_dist,avg_speed,occupancy,congestion_level,travel_time,accident_count,signal_timing,road_capacity
0,NL-007,Kingsway @ Park,28.647903,77.149142,WS-NORTH,2025-02-26,07:30,706.0,706,"{""2W"": 0.37, ""Car"": 0.43, ""LCV"": 0.11, ""HCV"": ...",31.2,59.6,Moderate,2.12,0,54,1928
1,NL-014,Industrial Link B,28.594749,77.253501,WS-CENTRAL,2025-03-23,21:00,308.0,308,"{""2W"": 0.43, ""Car"": 0.39, ""LCV"": 0.08, ""HCV"": ...",53.5,16.8,Free-flow,2.03,0,32,2327
2,NL-022,Old Town @ Clock,28.674521,77.219835,WS-NORTH,2025-04-05,11:00,342.0,342,"{""2W"": 0.4, ""Car"": 0.33, ""LCV"": 0.15, ""HCV"": 0...",49.9,27.5,Free-flow,3.46,0,38,1771
3,NL-006,Central Flyover E,28.622342,77.258257,WS-SOUTH,2025-05-04,01:30,187.0,187,"{""2W"": 0.41, ""Car"": 0.39, ""LCV"": 0.11, ""HCV"": ...",55.1,10.7,Free-flow,2.18,0,30,2285
4,NL-014,Industrial Link B,28.594749,77.253501,WS-CENTRAL,2025-01-08,12:30,490.0,490,"{""2W"": 0.39, ""Car"": 0.4, ""LCV"": 0.1, ""HCV"": 0.11}",45.9,27.7,Free-flow,2.37,0,45,2327


In [4]:
# --- Temporal coverage ---
t["timestamp"] = pd.to_datetime(t["date"] + " " + t["time"])
print("Date range:", t["timestamp"].min(), "to", t["timestamp"].max())
print("Unique dates:", t["date"].nunique())



Date range: 2025-01-01 00:00:00 to 2025-05-31 23:30:00
Unique dates: 151


In [5]:
# expected: 25 roads × ~151 days × 48 half-hour windows/day
expected_rows = t["road_id"].nunique() * t["date"].nunique() * 48
print("Expected rows if fully complete:", expected_rows)
print("Actual rows:", len(t))
print("Missing windows (sensor dropout):", expected_rows - len(t))

Expected rows if fully complete: 181200
Actual rows: 178468
Missing windows (sensor dropout): 2732


In [6]:
# --- Duplicates ---
dupe_count = t.duplicated(subset=["road_id", "date", "time"]).sum()
print("Key-level duplicates:", dupe_count)
t[t.duplicated(subset=["road_id", "date", "time"], keep=False)].sort_values(
    ["road_id", "date", "time"]
).head(10)

Key-level duplicates: 1767


,road_id,road_name,latitude,longitude,weather_station_id,date,time,traffic_volume,vehicle_count,vehicle_type_dist,avg_speed,occupancy,congestion_level,travel_time,accident_count,signal_timing,road_capacity,timestamp
137573,NL-001,Northline Ave @ 5th,28.643422,77.190411,WS-NORTH,2025-01-04,06:00,240.0,240,"{""2W"": 0.43, ""Car"": 0.35, ""LCV"": 0.11, ""HCV"": ...",55.2,15.8,Free-flow,2.16,0,36,1888,2025-01-04 06:00:00
147678,NL-001,Northline Ave @ 5th,28.643422,77.190411,WS-NORTH,2025-01-04,06:00,240.0,240,"{""2W"": 0.43, ""Car"": 0.35, ""LCV"": 0.11, ""HCV"": ...",55.2,15.8,Free-flow,2.16,0,36,1888,2025-01-04 06:00:00
19191,NL-001,Northline Ave @ 5th,28.643422,77.190411,WS-NORTH,2025-01-11,05:00,139.0,139,"{""2W"": 0.4, ""Car"": 0.37, ""LCV"": 0.13, ""HCV"": 0.1}",52.7,8.9,NaN,2.27,0,43,1888,2025-01-11 05:00:00
110142,NL-001,Northline Ave @ 5th,28.643422,77.190411,WS-NORTH,2025-01-11,05:00,139.0,139,"{""2W"": 0.4, ""Car"": 0.37, ""LCV"": 0.13, ""HCV"": 0.1}",52.7,8.9,NaN,2.27,0,43,1888,2025-01-11 05:00:00
59694,NL-001,Northline Ave @ 5th,28.643422,77.190411,WS-NORTH,2025-01-11,09:30,NaN,641,"{""2W"": 0.38, ""Car"": 0.44, ""LCV"": 0.11, ""HCV"": ...",NaN,NaN,Moderate,2.96,0,59,1888,2025-01-11 09:30:00
70453,NL-001,Northline Ave @ 5th,28.643422,77.190411,WS-NORTH,2025-01-11,09:30,NaN,641,"{""2W"": 0.38, ""Car"": 0.44, ""LCV"": 0.11, ""HCV"": ...",NaN,NaN,Moderate,2.96,0,59,1888,2025-01-11 09:30:00
26562,NL-001,Northline Ave @ 5th,28.643422,77.190411,WS-NORTH,2025-01-12,06:00,208.0,208,"{""2W"": 0.38, ""Car"": 0.37, ""LCV"": 0.13, ""HCV"": ...",50.9,13.5,Free-flow,2.35,0,40,1888,2025-01-12 06:00:00
36768,NL-001,Northline Ave @ 5th,28.643422,77.190411,WS-NORTH,2025-01-12,06:00,208.0,208,"{""2W"": 0.38, ""Car"": 0.37, ""LCV"": 0.13, ""HCV"": ...",50.9,13.5,Free-flow,2.35,0,40,1888,2025-01-12 06:00:00
152380,NL-001,Northline Ave @ 5th,28.643422,77.190411,WS-NORTH,2025-01-14,05:30,224.0,224,"{""2W"": 0.42, ""Car"": 0.39, ""LCV"": 0.09, ""HCV"": ...",51.3,16.4,NaN,2.33,0,39,1888,2025-01-14 05:30:00
152897,NL-001,Northline Ave @ 5th,28.643422,77.190411,WS-NORTH,2025-01-14,05:30,224.0,224,"{""2W"": 0.42, ""Car"": 0.39, ""LCV"": 0.09, ""HCV"": ...",51.3,16.4,NaN,2.33,0,39,1888,2025-01-14 05:30:00


In [7]:
# --- Missingness ---
missing = t.isnull().sum()
missing_pct = (missing / len(t) * 100).round(2)
pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})[missing > 0]

,missing_count,missing_pct
traffic_volume,4387,2.46
avg_speed,4382,2.46
occupancy,4383,2.46
congestion_level,26883,15.06


In [8]:
# --- Rule-based (impossible) outliers ---
outlier_report = {
    "negative_traffic_volume": (t["traffic_volume"] < 0).sum(),
    "avg_speed_over_200": (t["avg_speed"] > 200).sum(),
    "occupancy_over_100": (t["occupancy"] > 100).sum(),
    "negative_accident_count": (t["accident_count"] < 0).sum(),
}
outlier_report

{'negative_traffic_volume': np.int64(241),
 'avg_speed_over_200': np.int64(237),
 'occupancy_over_100': np.int64(234),
 'negative_accident_count': np.int64(0)}

In [9]:
# --- Statistical outliers (IQR method) on legitimate readings only ---
def iqr_outliers(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((series < lower) | (series > upper)).sum()

for col in ["traffic_volume", "avg_speed", "occupancy", "travel_time"]:
    print(f"{col}: {iqr_outliers(t[col].dropna())} statistical outliers (IQR)")

traffic_volume: 2151 statistical outliers (IQR)
avg_speed: 4999 statistical outliers (IQR)
occupancy: 1228 statistical outliers (IQR)
travel_time: 11113 statistical outliers (IQR)


In [10]:
# --- Target variable distributions ---
t["traffic_volume"].describe()

count    174081.000000
mean        430.618970
std         283.745371
min         -59.000000
25%         195.000000
50%         361.000000
75%         600.000000
max        2090.000000
Name: traffic_volume, dtype: float64

In [11]:
t["congestion_level"].value_counts(dropna=False, normalize=True) * 100


congestion_level
Free-flow    52.126432
Moderate     20.278705
NaN          15.063205
Heavy         7.849026
Severe        4.682632
Name: proportion, dtype: float64

In [12]:
# --- Per-road summary (useful sanity check + README table) ---
t.groupby("road_id").agg(
    n_readings=("traffic_volume", "count"),
    avg_volume=("traffic_volume", "mean"),
    avg_speed=("avg_speed", "mean"),
).round(1).sort_values("avg_volume", ascending=False)

,n_readings,avg_volume,avg_speed
road_id,,,
NL-006,6953,563.4,41.1
NL-019,6924,562.3,41.4
NL-004,6996,537.5,41.1
NL-002,6982,536.2,42.2
NL-005,6951,510.2,41.1
NL-023,6963,503.0,42.9
NL-016,6935,491.0,42.5
NL-014,6962,488.5,43.7
NL-015,6956,483.6,41.3


In [17]:
# --- Parse the nested JSON field ---
import json
sample = t["vehicle_type_dist"].dropna().iloc[0]
print(sample, "→", json.loads(sample.replace('""', '"')) if '""' in sample else json.loads(sample))

{"2W": 0.37, "Car": 0.43, "LCV": 0.11, "HCV": 0.09} → {'2W': 0.37, 'Car': 0.43, 'LCV': 0.11, 'HCV': 0.09}


*Weather_Conditions*

In [18]:
w = pd.read_csv("../data/raw/weather_observations.csv")
w["timestamp"] = pd.to_datetime(w["date"], format="%d/%m/%Y") + pd.to_timedelta(
    w["time"] + ":00"
)

print(w.shape)
print("Date range:", w["timestamp"].min(), "to", w["timestamp"].max())
print("Stations:", w["station_id"].unique())

(10872, 8)
Date range: 2025-01-01 00:00:00 to 2025-05-31 23:00:00
Stations: <StringArray>
['WS-NORTH', 'WS-CENTRAL', 'WS-SOUTH']
Length: 3, dtype: str


In [19]:
# --- The label inconsistency problem ---
print("Raw unique values:", sorted(w["weather_condition"].dropna().unique()))
print("\nValue counts:")
w["weather_condition"].value_counts()

Raw unique values: ['CLEAR', 'Clear', 'Cloudy', 'FOG', 'Fog', 'Overcast', 'RAIN', 'Rain', 'clear', 'cloudy', 'foggy', 'rain ', 'rainy']

Value counts:


weather_condition
Clear       5647
Cloudy      1504
clear       1284
CLEAR       1237
Overcast     358
cloudy       340
Rain         307
Fog           59
rain          48
rainy         37
RAIN          34
foggy         10
FOG            7
Name: count, dtype: int64

In [20]:
# --- Missingness ---
w.isnull().sum()

station_id             0
date                   0
time                   0
weather_condition      0
temperature          167
rainfall               0
visibility           111
timestamp              0
dtype: int64

In [21]:
# --- Distributions ---
w[["temperature", "rainfall", "visibility"]].describe()

,temperature,rainfall,visibility
count,10705.000000,10872.000000,10761.000000
mean,17.980318,0.190664,9674.190874
std,4.376441,1.056368,1499.340546
min,5.800000,0.000000,301.000000
25%,14.600000,0.000000,10000.000000
50%,18.000000,0.000000,10000.000000
75%,21.300000,0.000000,10000.000000
max,29.800000,8.900000,10000.000000


In [22]:
# --- Coverage check per station ---
w.groupby("station_id")["timestamp"].agg(["min", "max", "count"])

,min,max,count
station_id,,,
WS-CENTRAL,2025-01-01,2025-05-31 23:00:00,3624
WS-NORTH,2025-01-01,2025-05-31 23:00:00,3624
WS-SOUTH,2025-01-01,2025-05-31 23:00:00,3624


*Calendar_events*

In [23]:
c = pd.read_csv("../data/raw/calendar_events.csv")
c["date"] = pd.to_datetime(c["date"])

print(c.shape)
print("Date range:", c["date"].min(), "to", c["date"].max())
print("Public holidays:", c["public_holiday"].sum())
print("Event days:", c["event_flag"].sum())
print("Roadwork days:", c["roadwork_flag"].sum())

(151, 6)
Date range: 2025-01-01 00:00:00 to 2025-05-31 00:00:00
Public holidays: 6
Event days: 6
Roadwork days: 11


In [24]:
c[c["public_holiday"] == 1][["date", "holiday_name"]]

,date,holiday_name
0,2025-01-01,New Year's Day
25,2025-01-26,Republic Day
45,2025-02-15,Regional Festival
73,2025-03-15,Spring Holiday
119,2025-04-30,Labour Day
130,2025-05-11,City Foundation Day


In [25]:
c[c["event_flag"] == 1][["date", "event_name"]]

,date,event_name
12,2025-01-13,Marathon Road Closure
33,2025-02-03,Stadium Concert
58,2025-02-28,Trade Expo
88,2025-03-30,Cricket Fixture
104,2025-04-15,Political Rally
141,2025-05-22,Music Festival


**Checking Cross File Consistency**

In [26]:
# Do all traffic weather_station_id values exist in the weather table?
traffic_stations = set(t["weather_station_id"].unique())
weather_stations = set(w["station_id"].unique())
print("In traffic but not weather:", traffic_stations - weather_stations)
print("In weather but not traffic:", weather_stations - traffic_stations)

In traffic but not weather: set()
In weather but not traffic: set()


That's exactly what both sets are empty (set()), means

1. Every weather_station_id in the traffic table has a matching station_id in the weather table.
2. No weather station exists that traffic never references.

In [27]:
# Do the three files cover the same date range?
print("Traffic dates:", t["date"].min(), "-", t["date"].max())
print("Weather dates:", w["timestamp"].dt.date.min(), "-", w["timestamp"].dt.date.max())
print("Calendar dates:", c["date"].min().date(), "-", c["date"].max().date())

Traffic dates: 2025-01-01 - 2025-05-31
Weather dates: 2025-01-01 - 2025-05-31
Calendar dates: 2025-01-01 - 2025-05-31
